# AIE464 — Kaggle Smoke Notebook
ดัดแปลงการตั้งค่า environment จาก `03_final_benchmark_colab.ipynb` ใช้ source/models/loss/split เดิมทั้งหมด

**รออนุมัติ GPU Smoke Test ก่อนรัน** Notebook นี้ล็อก `mode="smoke"`, `epochs=1`: 1 optimizer update ต่อโมเดล และ Validation สองหน้าเต็มจากสองกลุ่มตาม protocol เดิม

ไม่มี Full Training และไม่มีการประเมิน Test Set แม้เปลี่ยน flags จะถูกปฏิเสธ ต้องได้รับอนุมัติและปรับ Notebook ในขั้นตอนใหม่ก่อนขยายขอบเขต

เพิ่ม Dataset `thames98451/aie464-denoising-benchmark` ใน Kaggle และเลือก GPU ใช้ GPU 0 เพียงตัวเดียว ไม่เปลี่ยน batch size หรือวิธีทดลอง


In [ ]:
import os, sys
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.dont_write_bytecode = True  # Do not write __pycache__ into read-only Input.
from pathlib import Path
import json, hashlib, zipfile, math, uuid
from datetime import datetime, timezone

PROJECT_DIR = Path('/kaggle/input/datasets/thames98451/aie464-denoising-benchmark/mini_project')
WORKING_DIR = Path('/kaggle/working')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Attach the existing benchmark Dataset: {PROJECT_DIR}')
if not PROJECT_DIR.resolve().is_relative_to(Path('/kaggle/input').resolve()):
    raise ValueError('Source and data must be read from Kaggle Input')
if not WORKING_DIR.is_dir():
    raise FileNotFoundError('This notebook must run in Kaggle')

EXPECTED_SOURCE_HASHES = {
    "dataset.py": "ab2481e18f3168b618a6cb3282200336785aba2e737ccb56f99e61ffe8bd8134",
    "evaluate.py": "39014632456859ffcd9810b3663f3647cab2cabd0caa98a6f57b9f52cbb1b41a",
    "benchmark.py": "a88fbf9e71106a3e111bce344bdd4cb0674148ef83134e1ede1fe4c113ee8bf9",
    "train.py": "bd5fc026ed342e523b12a0ed86651b3e19d0a60653a25f4bfff87445704e0de2",
    "losses.py": "cefc6c2bf8e8442c69e972cc9022f59455e10ab84f90fb360cc079bea8c7d1bd",
    "models/shallow_cnn.py": "46d09daa2e6796985395e4874eeacdf632bbfccb548b3c37cd40f5a6bb3b0cd9",
    "models/dncnn.py": "79399e92ea6d87d06dd7cf2868f0afef5bf1acc530f09f0d284b686ae0d13bdb",
    "models/unet.py": "c855089306999fd5eed5f695a5caee706c47677249898aba1db5c2968b6ea2c7"
}
for name, expected in EXPECTED_SOURCE_HASHES.items():
    actual = hashlib.sha256((PROJECT_DIR / 'src' / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Source differs from the approved pipeline: {name}. Update the Input Dataset; do not edit the pipeline here.')

sys.path.insert(0, str(PROJECT_DIR / 'src'))
import torch, numpy, PIL, matplotlib
from train import TrainConfig, train_experiment, source_hashes
from benchmark import EXPERIMENTS, load_manifest, json_save, MEMORY_LIMIT
# Reject stale imports from another project in a reused kernel.
for name in ('train', 'benchmark', 'dataset', 'evaluate', 'losses',
             'models.shallow_cnn', 'models.dncnn', 'models.unet'):
    if not Path(sys.modules[name].__file__).resolve().is_relative_to((PROJECT_DIR / 'src').resolve()):
        raise RuntimeError('Restart the kernel: source modules were imported from another directory')
manifest = load_manifest(PROJECT_DIR)  # Metadata only, including split separation.
print('Frozen split counts:', manifest['counts'])


## Approval gates และ Smoke configuration
Full Training/Final Test ปิดและถูกล็อกใน Notebook นี้ แก้ RUN_ID สำหรับ run ใหม่เท่านั้น
Resume ใช้ RUN_ID/config/source/runtime เดิมเพื่อไม่ให้เพิ่ม optimizer updates หากโมเดลนั้นเสร็จแล้ว


In [ ]:
GPU_SMOKE_APPROVED = False  # Set True only after explicit approval.
RUN_FULL_TRAINING = False
TRAINING_APPROVED = False
RUN_FINAL_TEST = False
FINAL_TEST_APPROVED = False
MODE, EPOCHS = 'smoke', 1
RUN_ID = 'kaggle_smoke_20261013_01'
RESUME = False
RESTORE_ARCHIVE = None  # Optional Path('/kaggle/input/<private-backup>/<snapshot>.zip')
EXPECTED_BACKUP_SHA256 = ''  # Required when RESTORE_ARCHIVE is set; use downloaded .sha256.


def enforce_smoke_scope():
    if RUN_FULL_TRAINING or TRAINING_APPROVED:
        raise PermissionError('Full Training is locked; separate approval and notebook revision required')
    if RUN_FINAL_TEST or FINAL_TEST_APPROVED:
        raise PermissionError('Final Test is locked; separate approval and notebook revision required')
    if MODE != 'smoke' or EPOCHS != 1:
        raise ValueError('This notebook permits exactly one smoke update per model')


def require_gpu_smoke_approval():
    enforce_smoke_scope()
    if not GPU_SMOKE_APPROVED:
        raise PermissionError('Wait for GPU Smoke approval before running GPU cells')


enforce_smoke_scope()
if Path(RUN_ID).name != RUN_ID or RUN_ID in ('', '.', '..'):
    raise ValueError('RUN_ID must be one folder name')
RUN_DIR = WORKING_DIR / 'final_benchmark' / RUN_ID
BACKUP_DIR = WORKING_DIR / 'backups'
print('Read-only project:', PROJECT_DIR)
print('Results:', RUN_DIR)
print('ZIP snapshots:', BACKUP_DIR)


## สำรองผลและกู้คืน
สร้าง ZIP + SHA256 หลังแต่ละโมเดล รวมเฉพาะ checkpoints/logs/ภาพของ run นี้ ไม่มี Dataset/source/secrets
ดาวน์โหลด ZIP และ `.sha256` จาก `/kaggle/working/backups/` ไปเครื่องหรือ Drive **ก่อนปิด Session**
ZIP ที่ยังอยู่ใน `/kaggle/working/` อย่างเดียวไม่ใช่การสำรองนอก Session

กู้ Session ใหม่: เพิ่ม ZIP ที่ดาวน์โหลดไว้เป็น Kaggle Input ส่วนตัว ตั้ง RESTORE_ARCHIVE,
EXPECTED_BACKUP_SHA256, RESUME=True และ RUN_ID เดิม ระบบตรวจ ZIP และ hashes ก่อนคัดลอกกลับไป Working
Resume เดิมจะปฏิเสธหาก software/GPU versions, config, source หรือข้อมูลเปลี่ยน


In [ ]:
ARTIFACT_NAMES = ('best.pt', 'last.pt', 'config.json', 'step_log.json',
                  'validation.json', 'metrics.json', 'loss.png', 'comparisons.png')
RUN_METADATA = ('kaggle_preflight.json', 'smoke_verification.json')


def allowed_backup_names():
    return {f'{RUN_ID}/{experiment}/{name}' for experiment in EXPERIMENTS for name in ARTIFACT_NAMES} | {
        f'{RUN_ID}/{name}' for name in RUN_METADATA}


def backup_run():
    records = {}
    for name in sorted(allowed_backup_names()):
        path = RUN_DIR.parent / name
        if path.is_file():
            if path.is_symlink():
                raise ValueError('Do not archive symlinks')
            records[name] = {'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                             'bytes': path.stat().st_size}
    if not records:
        return None
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    archive = BACKUP_DIR / f'{RUN_ID}_{stamp}_{uuid.uuid4().hex[:8]}.zip'
    metadata = {'run_id': RUN_ID, 'mode': 'smoke', 'epochs': 1,
                'source_hashes': EXPECTED_SOURCE_HASHES, 'files': records}
    with zipfile.ZipFile(archive, mode='x', compression=zipfile.ZIP_DEFLATED) as bundle:
        for name in records:
            bundle.write(RUN_DIR.parent / name, name)
        bundle.writestr('backup_manifest.json', json.dumps(metadata, indent=2))
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise RuntimeError('Backup ZIP integrity check failed')
    checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
    sidecar = archive.with_suffix('.zip.sha256')
    sidecar.write_text(f'{checksum}  {archive.name}\n')
    print('Download this snapshot + checksum before closing the session:', archive, sidecar)
    return archive


def restore_backup(archive, checksum):
    archive = Path(archive)
    if RUN_DIR.exists():
        raise FileExistsError('Restore never overwrites an existing run')
    if len(checksum) != 64 or hashlib.sha256(archive.read_bytes()).hexdigest() != checksum:
        raise ValueError('Provide the matching external backup SHA256')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or bundle.testzip() is not None:
            raise ValueError('Invalid backup ZIP')
        metadata = json.loads(bundle.read('backup_manifest.json'))
        records = metadata['files']
        if (metadata['run_id'] != RUN_ID or metadata['mode'] != 'smoke' or metadata['epochs'] != 1
                or metadata['source_hashes'] != EXPECTED_SOURCE_HASHES):
            raise ValueError('Backup protocol differs')
        if not set(records).issubset(allowed_backup_names()) or set(names) != set(records) | {'backup_manifest.json'}:
            raise ValueError('Backup contains unexpected files')
        # Validate every path and content before writing any restored file.
        for name, record in records.items():
            destination = (RUN_DIR.parent / name).resolve()
            content = bundle.read(name)
            if (not destination.is_relative_to(RUN_DIR.resolve()) or len(content) != record['bytes']
                    or hashlib.sha256(content).hexdigest() != record['sha256']):
                raise ValueError('Backup path or contents differ')
        for name in records:
            destination = RUN_DIR.parent / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(bundle.read(name))
    print('Restored to Working; the unchanged pipeline will verify Resume provenance.')


if RESTORE_ARCHIVE is not None:
    if not RESUME:
        raise ValueError('Set RESUME=True to use a restored run')
    restore_backup(RESTORE_ARCHIVE, EXPECTED_BACKUP_SHA256)
if RESUME and not RUN_DIR.is_dir():
    raise FileNotFoundError('Restore your external backup or select an existing run before Resume')
if not RESUME and RUN_DIR.exists():
    raise FileExistsError('Choose a new RUN_ID; never overwrite an existing run')


## CUDA/AMP preflight
AMP คือการคำนวณบางส่วนด้วย float16 เพื่อลดหน่วยความจำ จะทดสอบ CUDA matrix operation
และ GradScaler ก่อน Smoke จริง ใช้ GPU 0 เท่านั้นและหยุดหากเกินงบ 6 GiB


In [ ]:
require_gpu_smoke_approval()
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator; no CPU fallback')
torch.cuda.set_device(0)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)
with torch.inference_mode(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
    probe = torch.ones(16, 16, device='cuda') @ torch.ones(16, 16, device='cuda')
if probe.dtype != torch.float16 or not torch.isfinite(probe).all():
    raise RuntimeError('CUDA float16 autocast probe failed')
scaler_probe = torch.amp.GradScaler('cuda')
if not scaler_probe.is_enabled():
    raise RuntimeError('CUDA GradScaler unavailable')
torch.cuda.synchronize(0)
preflight = {'device': 'cuda:0', 'gpu': torch.cuda.get_device_name(0),
             'gpu_count': torch.cuda.device_count(), 'torch': str(torch.__version__),
             'cuda': torch.version.cuda, 'amp_probe_dtype': str(probe.dtype),
             'grad_scaler_enabled': scaler_probe.is_enabled(),
             'source_hashes': source_hashes(PROJECT_DIR), 'mode': MODE, 'epochs': EPOCHS,
             'test_evaluated': False}
del probe, scaler_probe
RUN_DIR.mkdir(parents=True, exist_ok=True)
json_save(preflight, RUN_DIR / 'kaggle_preflight.json')
print(preflight)


## GPU Smoke — ต้องได้รับอนุมัติก่อนรัน cell นี้
ใช้ `train_experiment` เดิม: forward/backward บน train crop ที่จับคู่ตำแหน่งกัน,
AMP/Adam/L1 หรือ Sobel ตาม experiment, Val PSNR/SSIM, checkpoint และภาพ
1 update ต่อโมเดลเท่านั้น มี automatic snapshot หลังแต่ละโมเดลและเมื่อ cell เกิดข้อผิดพลาด


In [ ]:
require_gpu_smoke_approval()
summaries = {}
for experiment in EXPERIMENTS:
    folder = RUN_DIR / experiment
    config = TrainConfig(experiment=experiment, mode='smoke', epochs=1)
    try:
        summaries[experiment] = train_experiment(
            PROJECT_DIR, folder, config,
            resume=RESUME and folder.exists(),
            approved_training=False,
            require_cuda=True,
        )
        summary = summaries[experiment]
        if summary['steps'] != 1 or summary['test_evaluated']:
            raise RuntimeError('Smoke scope or optimizer budget violated')
        print(experiment, summary['validation']['mean'], summary['gpu_memory'])
    finally:
        # This is still session-local until you download/save the output.
        backup_run()


In [ ]:
require_gpu_smoke_approval()
checks = {}
sequence = []
initial_unet = {}
for experiment in EXPERIMENTS:
    folder = RUN_DIR / experiment
    checkpoint = torch.load(folder / 'last.pt', map_location='cpu', weights_only=True)
    summary = summaries[experiment]
    history = checkpoint['history']
    if (checkpoint['completed_epochs'] != 1 or len(history) != 1
            or checkpoint['config']['mode'] != 'smoke' or checkpoint['test_evaluated']):
        raise RuntimeError('Checkpoint is not a one-update smoke checkpoint')
    if not all(math.isfinite(history[0][key]) for key in ('total', 'l1', 'edge', 'grad_norm')):
        raise RuntimeError('Non-finite forward/backward evidence')
    if history[0]['id'] not in manifest['splits']['train']:
        raise RuntimeError('Training ID outside training split')
    val_rows = summary['validation']['rows']
    if len(val_rows) != 2 or any(row['id'] not in manifest['splits']['val'] for row in val_rows):
        raise RuntimeError('Unexpected validation coverage')
    if max(summary['gpu_memory'].values()) > MEMORY_LIMIT:
        raise RuntimeError('GPU memory exceeded 6 GiB')
    if summary['versions']['device'] != 'cuda' or summary['versions']['cuda'] is None:
        raise RuntimeError('Smoke did not use CUDA')
    for name in ARTIFACT_NAMES:
        if not (folder / name).is_file() or (folder / name).stat().st_size == 0:
            raise RuntimeError(f'Missing saved artifact: {name}')
    sequence.append((history[0]['id'], tuple(history[0]['crop'])))
    if experiment.startswith('unet_'):
        initial_unet[experiment] = checkpoint['initial_state_sha256']
    checks[experiment] = {'optimizer_updates': 1, 'validation_pairs': 2,
                          'finite_forward_backward': True, 'checkpoint_reload': 'verified by unchanged pipeline',
                          'gpu_memory': summary['gpu_memory'], 'validation': summary['validation']['mean']}
if len(set(sequence)) != 1 or len(set(initial_unet.values())) != 1:
    raise RuntimeError('Model page/crop sequences or U-Net initialization differ')
json_save({'checks': checks, 'same_training_id_crop': True, 'same_unet_initialization': True,
           'amp_preflight': preflight, 'test_evaluated': False, 'quality_claim': False},
          RUN_DIR / 'smoke_verification.json')
latest_backup = backup_run()
print('Smoke pipeline verified; this is not proof of denoising quality.')


In [ ]:
from IPython.display import display, Image, FileLink
for experiment in EXPERIMENTS:
    print(experiment, '| Validation only: Noisy / Denoised / Clean')
    display(Image(filename=str(RUN_DIR / experiment / 'comparisons.png')))
    display(Image(filename=str(RUN_DIR / experiment / 'loss.png')))
print('Use Kaggle Output / file browser to download the ZIP and .sha256 if links are unavailable.')
display(FileLink(str(latest_backup)))
display(FileLink(str(latest_backup.with_suffix('.zip.sha256'))))


## ก่อนจบ Session
ดาวน์โหลด snapshot ZIP + `.sha256` ไปเครื่อง/Google Drive และตรวจว่า ZIP เปิดได้จริง
หรือบันทึก Notebook Version พร้อม outputs แล้วตรวจในหน้า Output ว่า checkpoints/ZIP มีครบ
การบันทึก source notebook อย่างเดียวไม่ใช่การสำรอง checkpoints

อย่ากด Save & Run All ตอนนี้: เป็นการรัน Notebook ใหม่ ต้องรออนุมัติ GPU Smoke ก่อน
และไม่จำเป็นต้องรันซ้ำเพียงเพื่อดาวน์โหลด ZIP ใน interactive session

คู่มือ: `docs/KAGGLE_SMOKE.md`; [Kaggle Notebooks documentation](https://www.kaggle.com/docs/notebooks)
Full Training และ Final Test ยังล็อกไว้ หยุดหลังเก็บผล Smoke เพื่อขออนุมัติขั้นตอนถัดไป
